## Load Data

In [62]:
import pandas as pd
import numpy as np

df_cc_calls = pd.read_csv('../../../data/interim/cc_calls_cleaned.csv')
print(f"Shape: {df_cc_calls.shape}")
df_cc_calls.head()

Shape: (31636, 30)


,contact_id,call_date,direction,cc_care_package,cc_care_package_discussed,cc_urgency_getting_on_site,cc_external_consultant,cc_agent_cross_sell_attempt,cc_customer_issues_concerns,cc_business_struggles_financial_hardship,...,cc_contractor_sentiment,cc_contractor_sentiment_start_score,cc_contractor_sentiment_end_score,cc_contractor_sentiment_overall_score,cc_pricing_mentioned,cc_pricing_sentiment_impact,cc_refund_discussed,cc_contractor_suggest_leave,cc_contractor_complained,co_ref
0,6.255130e+11,2025-05-08,OUT_BOUND,Standard,Yes,No,No,No,Yes,Yes,...,Dissatisfied,20.0,30.0,30.0,Yes,Yes,No,Yes,Yes,HV3323
1,5.910870e+11,2024-11-25,OUT_BOUND,Standard,Yes,No,No,No,Yes,No,...,Dissatisfied,0.0,0.0,0.0,Yes,Yes,No,Yes,Yes,PJ7066
2,5.650910e+11,2024-10-23,IN_BOUND,Standard,Yes,No,No,No,Yes,No,...,Dissatisfied,20.0,60.0,40.0,Yes,Yes,No,Yes,Yes,DP6030
3,5.939750e+11,2025-01-13,IN_BOUND,Premier,Yes,No,No,No,Yes,Yes,...,Dissatisfied,20.0,60.0,40.0,Yes,Yes,Yes,Yes,Yes,AM2413
4,6.222820e+11,2025-03-19,IN_BOUND,Standard,Yes,No,No,No,Yes,Yes,...,Dissatisfied,20.0,40.0,40.0,Yes,Yes,No,Yes,Yes,ED6707


## Rename call_year → renewal_year

This ensures cc_calls can be joined to billings on co_ref + renewal_year.

In [63]:
# #rename call_year to renewal_year to match billings composite key
# df_cc_calls.rename(columns={'call_date': 'renewal_year'}, inplace=True)
# Convert to datetime 
df_cc_calls['call_date'] = pd.to_datetime(df_cc_calls['call_date'], errors='coerce')

# Extract year
df_cc_calls['renewal_year'] = df_cc_calls['call_date'].dt.year
print(df_cc_calls['renewal_year'].value_counts().sort_index())

renewal_year
2024     5365
2025    25588
2026      683
Name: count, dtype: int64


## Convert Yes/No/Unknown Columns to Binary

In [64]:
yes_no_cols = [
    'cc_care_package_discussed',
    'cc_urgency_getting_on_site',
    'cc_external_consultant',
    'cc_agent_cross_sell_attempt',
    'cc_customer_issues_concerns',
    'cc_business_struggles_financial_hardship',
    'cc_questionnaire_completion',
    'cc_chasing_response',
    'cc_issues_within_questionnaire',
    'cc_login_issues',
    'cc_platform_issues',
    'cc_dissatisfaction_time_to_complete',
    'cc_process_complexity_concerns',
    'cc_questions_harder_than_expected',
    'cc_dissatisfaction_support',
    'cc_pricing_mentioned',
    'cc_pricing_sentiment_impact',
    'cc_refund_discussed',
    'cc_contractor_suggest_leave',
    'cc_contractor_complained',
]

for col in yes_no_cols:
    df_cc_calls[col] = df_cc_calls[col].map({'Yes': 1, 'No': 0, 'Unknown': 0})

## Encode Sentiment Column

In [65]:
sentiment_map = {
    'Satisfied'    :  1,
    'Neutral'      :  0,
    'Not Discussed':  0,
    'Dissatisfied' : -1,
    'Unknown'      :  0,
}

df_cc_calls['cc_sentiment_encoded'] = df_cc_calls['cc_contractor_sentiment'].map(sentiment_map)
df_cc_calls['cc_dissatisfied_flag'] = (df_cc_calls['cc_contractor_sentiment'] == 'Dissatisfied').astype(int)

## Encode Direction Column

In [66]:
# IN_BOUND = 1, OUT_BOUND = 0
df_cc_calls['is_inbound'] = (df_cc_calls['direction'] == 'IN_BOUND').astype(int)

## One Hot Encoding

In [67]:
ohe_cols = ['cc_care_package', 'cc_call_initiated_by']

for col in ohe_cols:
    df_cc_calls[col] = (
        df_cc_calls[col]
        .str.lower()
        .str.replace(' ', '_')
    )
    
df_cc_calls = pd.get_dummies(
    df_cc_calls,
    columns=ohe_cols,
    drop_first=False,
    dtype=int
)

## Feature Engineering

In [68]:
# Sentiment improved or worsened during the call
df_cc_calls['sentiment_improved'] = (
    df_cc_calls['cc_contractor_sentiment_end_score'] >
    df_cc_calls['cc_contractor_sentiment_start_score']
).astype(int)

df_cc_calls['sentiment_worsened'] = (
    df_cc_calls['cc_contractor_sentiment_end_score'] <
    df_cc_calls['cc_contractor_sentiment_start_score']
).astype(int)

# Sentiment change score
df_cc_calls['sentiment_change'] = (
    df_cc_calls['cc_contractor_sentiment_end_score'] -
    df_cc_calls['cc_contractor_sentiment_start_score']
)

# High risk call — multiple issues in one call
df_cc_calls['high_risk_call'] = (
    (df_cc_calls['cc_contractor_complained']               == 1) |
    (df_cc_calls['cc_contractor_suggest_leave']            == 1) |
    (df_cc_calls['cc_business_struggles_financial_hardship'] == 1) |
    (df_cc_calls['cc_refund_discussed']                    == 1)
).astype(int)

# Dissatisfaction issue count per call
dissatisfaction_cols = [
    'cc_dissatisfaction_time_to_complete',
    'cc_process_complexity_concerns',
    'cc_questions_harder_than_expected',
    'cc_dissatisfaction_support',
    'cc_platform_issues',
    'cc_login_issues',
]
df_cc_calls['dissatisfaction_issue_count'] = df_cc_calls[dissatisfaction_cols].sum(axis=1)

# Pricing pressure flag
df_cc_calls['pricing_pressure_flag'] = (
    (df_cc_calls['cc_pricing_mentioned']        == 1) &
    (df_cc_calls['cc_pricing_sentiment_impact'] == 1)
).astype(int)

In [69]:
df_cc_calls.columns

Index(['contact_id', 'call_date', 'direction', 'cc_care_package_discussed',
       'cc_urgency_getting_on_site', 'cc_external_consultant',
       'cc_agent_cross_sell_attempt', 'cc_customer_issues_concerns',
       'cc_business_struggles_financial_hardship',
       'cc_questionnaire_completion', 'cc_chasing_response',
       'cc_issues_within_questionnaire', 'cc_login_issues',
       'cc_platform_issues', 'cc_dissatisfaction_time_to_complete',
       'cc_process_complexity_concerns', 'cc_questions_harder_than_expected',
       'cc_dissatisfaction_support', 'cc_contractor_sentiment',
       'cc_contractor_sentiment_start_score',
       'cc_contractor_sentiment_end_score',
       'cc_contractor_sentiment_overall_score', 'cc_pricing_mentioned',
       'cc_pricing_sentiment_impact', 'cc_refund_discussed',
       'cc_contractor_suggest_leave', 'cc_contractor_complained', 'co_ref',
       'renewal_year', 'cc_sentiment_encoded', 'cc_dissatisfied_flag',
       'is_inbound', 'cc_care_package_as

## Aggregate per co_ref + renewal_year

 grouping by co_ref + renewal_year ensures each renewal period gets only its own CC call signals.

In [70]:
agg_dict = {
    # Call counts
    'contact_id' : 'count',
    'is_inbound' : 'sum',

    # Sentiment
    'cc_sentiment_encoded'  : 'mean',
    'cc_dissatisfied_flag' : 'max',
    'cc_contractor_sentiment_start_score' : 'mean',
    'cc_contractor_sentiment_end_score' : 'mean',
    'cc_contractor_sentiment_overall_score' : 'mean',

    # Issue flags
    'cc_customer_issues_concerns' : 'max',
    'cc_business_struggles_financial_hardship' : 'max',
    'cc_login_issues'  : 'max',
    'cc_platform_issues' : 'max',
    'cc_dissatisfaction_time_to_complete' : 'max',
    'cc_process_complexity_concerns' : 'max',
    'cc_questions_harder_than_expected' : 'max',
    'cc_dissatisfaction_support' : 'max',
    'cc_pricing_mentioned' : 'max',
    'cc_pricing_sentiment_impact' : 'max',
    'cc_refund_discussed' : 'max',
    'cc_contractor_suggest_leave' : 'max',
    'cc_contractor_complained' : 'max',
    'cc_urgency_getting_on_site' : 'max',
    'cc_chasing_response' : 'max',
    'cc_questionnaire_completion' : 'max',
    'cc_care_package_discussed' : 'max',
    'cc_external_consultant' : 'max',
    'cc_agent_cross_sell_attempt' : 'max',
    'cc_issues_within_questionnaire' : 'max',

    # Engineered features
    'sentiment_improved' : 'max',
    'sentiment_worsened' : 'max',
    'sentiment_change' : 'mean',
    'high_risk_call' : 'max',
    'dissatisfaction_issue_count' : 'sum',
    'pricing_pressure_flag' : 'max',

    # OHE — care package type
    'cc_care_package_assisted' : 'max',
    'cc_care_package_express' : 'max',
    'cc_care_package_not_discussed' : 'max',
    'cc_care_package_premier' : 'max',
    'cc_care_package_standard'  : 'max',
    'cc_care_package_unknown' : 'max',

    # OHE — call initiated by
    'cc_call_initiated_by_agent' : 'sum',
    'cc_call_initiated_by_customer' : 'sum',
    'cc_call_initiated_by_not_relevant' : 'sum',
    'cc_call_initiated_by_unknown' : 'sum',
}

# KEY FIX: group by co_ref + renewal_year (not just co_ref)
df_cc_calls_agg = df_cc_calls.groupby(['co_ref', 'renewal_year']).agg(agg_dict).reset_index()

# Rename columns clearly
df_cc_calls_agg.rename(columns={
    'contact_id' : 'cc_total_calls',
    'is_inbound' : 'cc_inbound_calls',
}, inplace=True)

print(f"Shape after aggregation: {df_cc_calls_agg.shape}")
df_cc_calls_agg.head()

Shape after aggregation: (17548, 45)


,co_ref,renewal_year,cc_total_calls,cc_inbound_calls,cc_sentiment_encoded,cc_dissatisfied_flag,cc_contractor_sentiment_start_score,cc_contractor_sentiment_end_score,cc_contractor_sentiment_overall_score,cc_customer_issues_concerns,...,cc_care_package_assisted,cc_care_package_express,cc_care_package_not_discussed,cc_care_package_premier,cc_care_package_standard,cc_care_package_unknown,cc_call_initiated_by_agent,cc_call_initiated_by_customer,cc_call_initiated_by_not_relevant,cc_call_initiated_by_unknown
0,AA0584,2025,1,0,0.0,0,50.0,50.0,80.0,0,...,0,0,0,0,1,0,1,0,0,0
1,AA0750,2025,1,0,1.0,0,50.0,90.0,90.0,0,...,0,0,1,0,0,0,1,0,0,0
2,AA0784,2025,2,0,0.5,0,50.0,65.0,80.0,0,...,0,0,1,0,0,0,0,2,0,0
3,AA0882,2025,1,1,0.0,0,50.0,80.0,70.0,0,...,0,0,1,0,0,0,0,1,0,0
4,AA0923,2025,1,1,1.0,0,50.0,90.0,90.0,0,...,0,0,1,0,0,0,0,1,0,0


## Add Outbound Calls Column

In [71]:
df_cc_calls_agg['cc_outbound_calls'] = (
    df_cc_calls_agg['cc_total_calls'] - df_cc_calls_agg['cc_inbound_calls']
)

## Final Check

In [72]:
print("Nulls:")
print(df_cc_calls_agg.isnull().sum()[df_cc_calls_agg.isnull().sum() > 0])

print(f"\nShape                        : {df_cc_calls_agg.shape}")
print(f"Unique co_ref                : {df_cc_calls_agg['co_ref'].nunique()}")
print(f"Unique co_ref + renewal_year : {df_cc_calls_agg.drop_duplicates(subset=['co_ref','renewal_year']).shape[0]}")
df_cc_calls_agg.describe()

Nulls:
Series([], dtype: int64)

Shape                        : (17548, 46)
Unique co_ref                : 14988
Unique co_ref + renewal_year : 17548


,renewal_year,cc_total_calls,cc_inbound_calls,cc_sentiment_encoded,cc_dissatisfied_flag,cc_contractor_sentiment_start_score,cc_contractor_sentiment_end_score,cc_contractor_sentiment_overall_score,cc_customer_issues_concerns,cc_business_struggles_financial_hardship,...,cc_care_package_express,cc_care_package_not_discussed,cc_care_package_premier,cc_care_package_standard,cc_care_package_unknown,cc_call_initiated_by_agent,cc_call_initiated_by_customer,cc_call_initiated_by_not_relevant,cc_call_initiated_by_unknown,cc_outbound_calls
count,17548.000000,17548.000000,17548.000000,17548.000000,17548.000000,17548.000000,17548.000000,17548.000000,17548.000000,17548.000000,...,17548.000000,17548.000000,17548.000000,17548.000000,17548.000000,17548.000000,17548.00000,17548.000000,17548.000000,17548.000000
mean,2024.826020,1.802827,0.418509,0.487625,0.047812,52.634007,79.355012,81.640854,0.157055,0.049977,...,0.119159,0.844712,0.021313,0.181445,0.012024,0.474128,1.22806,0.090267,0.010372,1.384317
std,0.466697,1.337369,0.925456,0.488905,0.213374,10.505362,13.788586,9.844902,0.363863,0.217904,...,0.323985,0.362190,0.144430,0.385398,0.108997,0.693013,1.27702,0.352233,0.106791,1.207397
min,2024.000000,1.000000,0.000000,-1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000
25%,2025.000000,1.000000,0.000000,0.000000,0.000000,50.000000,76.666667,80.000000,0.000000,0.000000,...,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,1.000000
50%,2025.000000,1.000000,0.000000,0.500000,0.000000,50.000000,80.000000,82.500000,0.000000,0.000000,...,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.00000,0.000000,0.000000,1.000000
75%,2025.000000,2.000000,1.000000,1.000000,0.000000,50.000000,90.000000,90.000000,0.000000,0.000000,...,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,2.00000,0.000000,0.000000,2.000000
max,2026.000000,51.000000,51.000000,1.000000,1.000000,100.000000,100.000000,100.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,8.000000,51.00000,9.000000,3.000000,15.000000


## Save

In [73]:
df_cc_calls_agg.to_csv('../../../data/interim/cc_calls_features.csv', index=False)


## Hypotheses

Hypothesis 1 → Customers with higher number of issues are more likely to churn

Hypothesis 2 → Customers with lower sentiment scores tend to churn

Hypothesis 3 → Customers who complain more frequently are more likely to churn

Hypothesis 4 → Customers with fewer interactions (calls) are more likely to churn

Hypothesis 5 → Negative experience over time (low sentiment + high issues) increases churn probability